In [2]:
import numpy as np

# ---- 制造一个奇异矩阵 ----
print("===== 制造奇异矩阵 =====")
A_singular = np.array([[1, 2, 3],
                       [4, 5, 6],
                       [7, 8, 9]])  # 第三行 = 第一行 + 第二行，所以奇异
b = np.array([1, 2, 3])

print("A_singular = \n", A_singular)
print(f"秩: {np.linalg.matrix_rank(A_singular)} (列数 = 3，所以秩 < 3，奇异)")
print(f"行列式: {np.linalg.det(A_singular):.2e} (≈0)")

# 尝试 solve：会报错
print("\n尝试 np.linalg.solve...")
try:
    x = np.linalg.solve(A_singular, b)
    print(f"解: {x}")
except np.linalg.LinAlgError as e:
    print(f"❌ 报错: {e}")

# ---- 解决方案1：用最小二乘（lstsq）找最优解 ----
print("\n===== 方案1：用最小二乘 =====")
x_lstsq, residuals, rank, s = np.linalg.lstsq(A_singular, b, rcond=None)
print(f"最小二乘解: {x_lstsq}")
if len(residuals) > 0:
    print(f"残差: {residuals[0]:.4f}")  # 这是 ||Ax - b||^2
else:
    print("残差: 0.0000 (方程有无穷多解，最小二乘解为精确解)")
print(f"用这个解，A@x = {A_singular @ x_lstsq} (最接近 b={b})")

# ---- 解决方案2：如果 A 本身没有冗余，只是形状不对（长方形） ----
print("\n===== 长方形矩阵：A 是 4x3 =====")
A_rect = np.array([[1, 2, 3],
                   [4, 5, 6],
                   [7, 8, 9],
                   [10, 11, 12]])  # 4行3列，不是方阵
b_rect = np.array([1, 2, 3, 4])

print(f"A_rect 形状: {A_rect.shape}")
print(f"np.linalg.solve 会报错（不是方阵）")

# 只能用 lstsq
x_rect = np.linalg.lstsq(A_rect, b_rect, rcond=None)[0]
print(f"最小二乘解: {x_rect}")
print(f"A_rect @ x = {A_rect @ x_rect} (最接近 b={b_rect})")

# ---- 解决方案3：用正则化（岭回归的雏形） ----
# 给 A^T A 加上一个小的对角矩阵 λI，确保可逆
# 这是岭回归 (Ridge Regression) 的核心思想
print("\n===== 方案3：正则化（岭回归） =====")
lambda_reg = 0.1
A_square = A_singular  # 用上面的奇异矩阵
# 解 (A^T A + λI) x = A^T b
ATA = A_square.T @ A_square
ATb = A_square.T @ b
A_reg = ATA + lambda_reg * np.eye(A_square.shape[1])
x_reg = np.linalg.solve(A_reg, ATb)
print(f"正则化解: {x_reg}")
print(f"A @ x_reg = {A_square @ x_reg} (虽然不完全等于 b，但很接近)")

===== 制造奇异矩阵 =====
A_singular = 
 [[1 2 3]
 [4 5 6]
 [7 8 9]]
秩: 2 (列数 = 3，所以秩 < 3，奇异)
行列式: 0.00e+00 (≈0)

尝试 np.linalg.solve...
❌ 报错: Singular matrix

===== 方案1：用最小二乘 =====
最小二乘解: [-0.05555556  0.11111111  0.27777778]
残差: 0.0000 (方程有无穷多解，最小二乘解为精确解)
用这个解，A@x = [1. 2. 3.] (最接近 b=[1 2 3])

===== 长方形矩阵：A 是 4x3 =====
A_rect 形状: (4, 3)
np.linalg.solve 会报错（不是方阵）
最小二乘解: [-0.05555556  0.11111111  0.27777778]
A_rect @ x = [1. 2. 3. 4.] (最接近 b=[1 2 3 4])

===== 方案3：正则化（岭回归） =====
正则化解: [-0.04255198  0.11233724  0.26722646]
A @ x_reg = [0.98380188 1.99483703 3.00587217] (虽然不完全等于 b，但很接近)
